# Resource Observation

This notebook accompany the lecture on [Resource Observation](https://linhbngo.github.io/csc478/09-resource-observation/)

| Role | Node | Count | Resources |
|------|------|-------|-----------|
| Control plane (`rke2-server`) | `node1` | 1 | 8 cores, 10 GB RAM |
| Worker (`rke2-agent`) | `node2` | 2 | 8 cores, 10 GB RAM |

Cluster traffic uses a private L2 dataplane (`192.168.1.0/24`). Ansible reaches the nodes over the FABRIC management network.

In [1]:
nodeCnt = 3   # number of nodes
coreCnt = 8   # number of CPUs per node
memSize = 10  # memory per node
diskSize = 50 # storage per node

## Import the FABlib Library

In [2]:
from pathlib import Path
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager
import sys

for _candidate in [Path.cwd(), *Path.cwd().parents]:
    if (_candidate / "utils" / "ansible.py").exists():
        sys.path.insert(0, str(_candidate))
        break
else:
    raise ImportError("Could not find repo root containing utils/ansible.py")

from utils.ansible import generate_rke2_playbooks

fablib = fablib_manager()
fablib.verify_and_configure()
fablib.create_ssh_tunnel_config(overwrite=False)

User: lngo@wcupa.edu bastion key is valid!
Configuration is valid
User: lngo@wcupa.edu bastion key is valid!
Configuration is valid
Please save the config!

SSH tunnel config created and zipped at: /home/fabric/work/fabric_config/fabric_ssh_tunnel_tools.tgz

Download Instructions:
Download your custom `fabric_ssh_tunnel_tools.tgz` file from the `fabric_config` folder.

Usage Instructions:
1. Unzip the archive and place the resulting `fabric_ssh_tunnel_tools/` folder somewhere accessible from your terminal.
2. Open a terminal window (on Windows, use PowerShell).
3. Use `cd` to navigate into the `fabric_ssh_tunnel_tools` folder.
4. In your terminal, run the SSH tunnel command generated by the next notebook cell.
    


In [3]:
# change the user name to your username
slicePrefix = f"ngo-{nodeCnt}-{coreCnt}-{memSize}-{diskSize}-" 
network_name = "ramnet"
nic_name = "ramnic"

# calculate resource requirements
minCores = nodeCnt * coreCnt * 2
minMem = nodeCnt * memSize * 2

In [4]:
# lauching site
from ipaddress import IPv4Network

resources = fablib.get_resources()
resources.update()
sites = resources.get_site_names()
selectedSite = ""

for site in sites:
    cores = resources.get_core_available(site)
    ram = resources.get_ram_available(site)
    if cores < minCores or ram < minMem:
        continue
    sliceName = slicePrefix + site

    # Clean up left-over slices with the same names
    existing = None
    for s in fablib.get_slices():
        if s.get_name() == sliceName:
            print(f"Deleting existing slice: {sliceName}")
            s.delete()
            break

    # Set up new slice
    slice = fablib.new_slice(name=sliceName)
    net = slice.add_l2network(name=network_name, subnet=IPv4Network("192.168.1.0/24"))
    
    for i in range(1, nodeCnt + 1):
        node = slice.add_node(name=f"node{i}",
                              site=site,
                              cores=coreCnt,ram=memSize,disk=diskSize,
                              image="default_ubuntu_22",)
        iface = node.add_component(model="NIC_Basic", name=nic_name).get_interfaces()[0]
        iface.set_mode("config")
        net.add_interface(iface)

    print("==========================================================================")
    try:
        print(f"Submitting slice at {site}...")
        grader_key = Path("/home/fabric/.ssh/grader_key.pub").read_text().strip()
        slice.submit(progress=False, wait=True,extra_ssh_keys=[grader_key])
    except Exception as e:
        print(f"{site}: slice submission failed with error {e}")
        try:
            slice.delete()
        except Exception:
            pass
        continue
            
    # Setup networking
    for i in range(1, nodeCnt + 1):
        node = slice.get_node(name=f"node{i}")
        iface = node.get_interface(network_name=network_name)
        iface.ip_link_up()
        iface.ip_addr_add(
            addr=f"192.168.1.{i}",
            subnet=IPv4Network("192.168.1.0/24"),
        )
        print(f"{node.get_name()} -> 192.168.1.{i}")

    slice.update()
    if slice.get_state() == "Closing":
        print(f"Need to find new site")
        continue
    else: 
        selectedSite = site
        print(f"Slice at {selectedSite} is stable: {slice.isStable()}")
        break

Submitting slice at GPN...
Running post boot config threads ...
Post boot config node3, Done! (0 sec)
Post boot config node2, Done! (0 sec)
Post boot config node1, Done! (0 sec)
Saving fablib data...  Done!
Running post boot config threads ...
Post boot config node1, Done! (0 sec)
Post boot config node3, Done! (0 sec)
Post boot config node2, Done! (0 sec)
Saving fablib data...  Done!
node1 -> 192.168.1.1
node2 -> 192.168.1.2
node3 -> 192.168.1.3
Slice at GPN is stable: True


In [5]:
from pathlib import Path

TEMPLATES_DIR = Path("templates").resolve()
PLAYBOOK_DIR = Path("playbook").resolve()
PLAYBOOK_DIR.mkdir(exist_ok=True)
assert (TEMPLATES_DIR / "host.yml.template").exists(), f"not found in {TEMPLATES_DIR}"

sliceName = slicePrefix + selectedSite
slice = fablib.get_slice(name=sliceName)

generate_rke2_playbooks(slice, fablib, network_name, templates_dir=TEMPLATES_DIR, playbook_dir=PLAYBOOK_DIR)

Wrote Ansible files from /home/fabric/csc478/00_rke2_ssh/templates -> /home/fabric/csc478/00_rke2_ssh/playbook


'all:\n  vars:\n    ansible_become: true\n    ansible_ssh_private_key_file: "/home/fabric/.ssh/slice_key"\n    ansible_ssh_common_args: "-F /home/fabric/work/fabric_config/ssh_config"\n    rke2_token: "fabric-rke2-cluster-token"\n    rke2_server_private_ip: "192.168.1.1"\n    dataplane_gateway: "192.168.1.254"\n\n  children:\n    rke2_servers:\n      hosts:\n        node1:\n          ansible_host: "2610:e0:a04c:fab2:f816:3eff:fe06:373c"\n          ansible_user: "ubuntu"\n          ansible_python_interpreter: "/usr/bin/python3"\n          private_ip: "192.168.1.1"\n\n    rke2_agents:\n      hosts:\n        node2:\n          ansible_host: "2610:e0:a04c:fab2:f816:3eff:fe57:c3ff"\n          ansible_user: "ubuntu"\n          ansible_python_interpreter: "/usr/bin/python3"\n          private_ip: "192.168.1.2"\n        node3:\n          ansible_host: "2610:e0:a04c:fab2:f816:3eff:fe64:5e72"\n          ansible_user: "ubuntu"\n          ansible_python_interpreter: "/usr/bin/python3"\n          pr

In [6]:
import subprocess

# Playbook: Host Prerequisites
Path("logs").mkdir(exist_ok=True)
log_path = Path("logs") / f"{selectedSite}-ansible-prereqs.log"
with open(log_path, "w") as log_file:
    print("==== Running prerequisite playbook ====")
    prereq_result = subprocess.run(["ansible-playbook","-i",
                                    "playbook/inventory.yml",
                                    "playbook/playbook-prereqs.yml",],
        stdout=log_file,stderr=subprocess.STDOUT,text=True,)

    if prereq_result.returncode != 0:
        print(f"{selectedSite}: prerequisite playbook failed")
    else: 
        print(f"Presequisites are installed")

==== Running prerequisite playbook ====
Presequisites are installed


In [7]:
# Playbook: NFS
Path("logs").mkdir(exist_ok=True)
log_path = Path("logs") / f"{selectedSite}-ansible-nfs.log"
with open(log_path, "w") as log_file:    
    print("==== Running NFS playbook ====")
    nfs_result = subprocess.run(["ansible-playbook","-i",
                                 "playbook/inventory.yml",
                                 "playbook/playbook-nfs.yml",],
        stdout=log_file,stderr=subprocess.STDOUT,text=True,)

    if nfs_result.returncode != 0:
        print(f"{selectedSite}: NFS playbook failed")
    else:
        print("NFS playbook is completed")

==== Running NFS playbook ====
NFS playbook is completed


In [10]:
# Playbook: RKE
Path("logs").mkdir(exist_ok=True)
log_path = Path("logs") / f"{selectedSite}-ansible-rke2.log"
with open(log_path, "w") as log_file: 
    print("==== Running RKE2 playbook ====")
    rke2_result = subprocess.run(["ansible-playbook","-i",
                                  "playbook/inventory.yml",
                                  "playbook/playbook-rke2.yml",],
        stdout=log_file,stderr=subprocess.STDOUT,text=True,)

    if rke2_result.returncode != 0:
        print(f"{selectedSite}: RKE2 playbook failed")
    else:
        print("RKE2 installation is completed")

==== Running RKE2 playbook ====
RKE2 installation is completed


In [11]:
# RKE2 Validation

server = slice.get_node("node1")
print("==== nodes ====")
stdout, stderr = server.execute("kubectl get nodes -o wide", quiet=True)
print(stdout)

print("==== nginx-demo ====")
stdout, stderr = server.execute("kubectl get pods -l app=nginx-demo -o wide", quiet=True)
print(stdout)

print("==== nginx-demo ====")
stdout, stderr = server.execute("kubectl get svc -l app=nginx-demo -o wide", quiet=True)
print(stdout)

print("==== curl via NodePort on dataplane ====")
stdout, stderr = server.execute("curl -s -o /dev/null -w '%{http_code}\n' http://192.168.1.1:30080/", quiet=True)
if stdout.strip() == "200":
    print(f"{selectedSite} works ...")
    print(f"Getting rid of nginx-demo ...")
    server.execute("kubectl delete deployment nginx-demo", quiet=True)
    server.execute("kubectl delete svc nginx-demo", quiet=True)

nodes = slice.get_nodes()
for node in nodes:
    print("----", node.get_name(), "----")
    print("management ip:", node.get_management_ip())
    print(node.get_ssh_command())

==== nodes ====
NAME    STATUS   ROLES                AGE   VERSION          INTERNAL-IP   EXTERNAL-IP   OS-IMAGE             KERNEL-VERSION               CONTAINER-RUNTIME
node1   Ready    control-plane,etcd   13h   v1.36.5+rke2r1   192.168.1.1   <none>        Ubuntu 22.04.5 LTS   5.15.0-185-generic (amd64)   containerd://2.3.4-k3s1
node2   Ready    <none>               13h   v1.36.5+rke2r1   192.168.1.2   <none>        Ubuntu 22.04.5 LTS   5.15.0-185-generic (amd64)   containerd://2.3.4-k3s1
node3   Ready    <none>               13h   v1.36.5+rke2r1   192.168.1.3   <none>        Ubuntu 22.04.5 LTS   5.15.0-185-generic (amd64)   containerd://2.3.4-k3s1

==== nginx-demo ====
NAME                         READY   STATUS    RESTARTS   AGE   IP          NODE    NOMINATED NODE   READINESS GATES
nginx-demo-6f8d7bb5d-bj4st   1/1     Running   0          13h   10.42.2.8   node3   <none>           <none>
nginx-demo-6f8d7bb5d-vxtlt   1/1     Running   0          13h   10.42.1.2   node2   <none> 

In [ ]:
slice.delete()